# 📌 Project 5 — Clip a Satellite Image

## 🎯 Goal

In this project, you will learn how to:

- Load a Sentinel-2 image.
- Display the original image.
- Clip the image to your study area.
- Compare the original and clipped images.

---

## 📚 Skills You'll Learn

- ✅ `clip()`
- ✅ Polygon geometries
- ✅ `filterBounds()`
- ✅ `Map.addLayer()`
- ✅ Image processing

---

# 🌍 Why Is Clipping Useful?

Imagine you're studying **Berlin**.

A Sentinel-2 image usually covers a much larger area than your study site.

```text
+--------------------------------------+
|                                      |
|      Entire Sentinel-2 Image         |
|                                      |
|        Berlin                        |
|          +--------+                  |
|          |        |                  |
|          | Study  |                  |
|          | Area   |                  |
|          +--------+                  |
|                                      |
+--------------------------------------+
```

Do you need the entire image?

**No.**

You only need the pixels inside your study area.

So instead of analyzing the whole image, we **clip** it.

After clipping:

```text
+--------+
|        |
| Study  |
| Area   |
|        |
+--------+
```

Everything outside the polygon becomes invisible.

---

# ✂️ What Does `clip()` Do?

Think of a pair of scissors.

Original paper:

```text
██████████████████
██████████████████
██████████████████
```

Cut around a shape:

```text
█████
█████
█████
```

The `clip()` function cuts an image using a geometry.

---

# Step 1 — Create a Study Area

This time we'll use a **Polygon** instead of a Point.

```javascript
var cityArea = ee.Geometry.Polygon([
  [
    [13.20, 52.60],
    [13.65, 52.60],
    [13.65, 52.35],
    [13.20, 52.35],
    [13.20, 52.60]
  ]
]);
```

## Why use a polygon?

A **Point** has no area.

```text
📍
```

You cannot meaningfully cut an image using a single point.

A **Polygon** represents an area.

```text
+---------+
|         |
|  Area   |
|         |
+---------+
```

This makes it suitable as a clipping boundary.

---

# Step 2 — Load a Sentinel-2 Image

```javascript
var image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(cityArea)
  .filterDate('2024-06-01', '2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first();
```

Nothing new here.

Notice that we now use:

```javascript
.filterBounds(cityArea)
```

instead of

```javascript
.filterBounds(cityPoint)
```

because our study area is now a polygon.

---

# Step 3 — Center the Map

```javascript
Map.centerObject(cityArea, 10);
```

## A useful observation

Previously we centered the map on a point:

```javascript
Map.centerObject(cityPoint, 10);
```

Now we're centering it on a polygon.

`Map.centerObject()` works with many Earth Engine objects, including:

- 📍 Points
- ⬜ Polygons
- 🛰 Images

It simply centers the map on the object you provide.

---

# Step 4 — Define Visualization Settings

```javascript
var trueColor = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000
};
```

You've already seen this visualization object in previous projects.

Creating it once allows you to reuse it whenever you display the image.

---

# Step 5 — Display the Original Image

```javascript
Map.addLayer(image, trueColor, "Original");
```

This displays the complete Sentinel-2 image.

---

# Step 6 — Clip the Image

Here's the new function.

```javascript
var clippedImage = image.clip(cityArea);
```

Let's break it down.

### `image`

The satellite image.

↓

### `clip()`

Cut the image.

↓

### `cityArea`

Use this polygon as the cutting boundary.

↓

### Result

A new image containing only the pixels inside the polygon.

---

## Remember

Earth Engine objects are **immutable**.

This code:

```javascript
var clippedImage = image.clip(cityArea);
```

does **not** change `image`.

Instead:

```text
Original Image
        ↓
     clip()
        ↓
   New Image
```

---

# Step 7 — Display the Clipped Image

```javascript
Map.addLayer(clippedImage, trueColor, "Clipped");
```

Now compare:

- 🌍 Original image
- ✂️ Clipped image

Turn each layer on and off in the **Layers** panel to see the difference.

---

# Complete Code

```javascript
// Study area
var cityArea = ee.Geometry.Polygon([
  [
    [13.20, 52.60],
    [13.65, 52.60],
    [13.65, 52.35],
    [13.20, 52.35],
    [13.20, 52.60]
  ]
]);

// Load image
var image = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
  .filterBounds(cityArea)
  .filterDate('2024-06-01', '2024-06-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first();

// Center map
Map.centerObject(cityArea, 10);

// Visualization
var trueColor = {
  bands: ['B4', 'B3', 'B2'],
  min: 0,
  max: 3000
};

// Display original image
Map.addLayer(image, trueColor, "Original");

// Clip image
var clippedImage = image.clip(cityArea);

// Display clipped image
Map.addLayer(clippedImage, trueColor, "Clipped");
```

---

# Workflow Summary

```text
Create Polygon
        ↓
Load Sentinel-2 Image
        ↓
Display Original Image
        ↓
Clip Image Using Polygon
        ↓
Display Clipped Image
```

---

# Understanding `clip()`

Think about what each part means:

```javascript
image.clip(cityArea)
```

| Part | Meaning |
|------|---------|
| `image` | What do I want to clip? |
| `clip()` | Perform the clipping operation |
| `cityArea` | Which geometry should be used as the boundary? |

---

## Before and After

### Before

```text
████████████████████████
████████████████████████
████████████████████████
```

### After

```text
      ███████
      ███████
      ███████
```

Only the pixels inside the polygon remain visible.

---

# Common Beginner Mistakes

## ❌ Clipping with a Point

```javascript
image.clip(cityPoint)
```

Although Earth Engine allows this, a point has virtually no area.

In practice, clipping is almost always done with a polygon or another area geometry.

---

## ❌ Thinking `clip()` Changes the Original Image

```javascript
var clipped = image.clip(cityArea);
```

This creates a **new image**.

The original image remains unchanged.

---

## ❌ Forgetting to Store the Result

This won't save the clipped image:

```javascript
image.clip(cityArea);
```

Instead, write:

```javascript
var clippedImage = image.clip(cityArea);
```

---

# Key Concepts

## ✅ `clip()`

Use `clip()` when you're interested in analyzing only a specific study area.

---

## ✅ `clip()` Returns a New Image

The original image is never modified.

---

## ✅ Geometry Defines **Where**

The geometry tells Earth Engine **where** the analysis should take place.

---

## ✅ Image Defines **What**

The image contains the satellite information you want to analyze.

Together:

```text
Geometry
      +
Image
      ↓
Spatial Analysis
```

This is one of the most important ideas in Google Earth Engine.

---

# Practice Exercise

Choose your own study area (for example, **Isfahan** or another city).

Your project should:

- ✅ Create a polygon.
- ✅ Load a Sentinel-2 image.
- ✅ Filter the collection using the polygon.
- ✅ Display the original image.
- ✅ Clip the image to the polygon.
- ✅ Display the clipped image.
- ✅ Use a reusable `trueColor` visualization object.

---

# 🚀 Bonus Challenge

Create a **False Color** version of the clipped image using:

```javascript
bands: ['B8', 'B4', 'B3']
```

Compare the True Color and False Color clipped images to see how vegetation appears in each visualization.